# Independence in Privacy Accounting

**Independence** means a user's privacy cost is charged only in proportion to their
personal contribution to each query result.

## Accountants compared

| Accountant | TCT rounds | Non-TCT rounds |
|---|---|---|
| Standard IDP | ε every round, every active user | ε every round, every active user |
| Adaptive TCT (independent) | Incremental charge only if user contributed to a top | ε only if user participated in that query |

Standard IDP has no notion of independence: every participating user pays ε regardless of their individual contribution.
Adaptive TCT charges only users who contributed to a target hit — cost concentrates on heavy hitters.

## Non-TCT query model

Each round, Poisson(λ) non-TCT queries fire. Each query has cost ~ Uniform(eps_min, eps_max).
Under Adaptive TCT only users who participated (p_participate probability) pay; under Standard IDP all active users pay.

**Sections:**
1. Parameters and setup
2. Single run — retention and budget curves
3. Effect of participation probability on the independence advantage
4. Per-user cost stratification by contribution count

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from simulation.query import compute_round_conversions, laplace_threshold_query
from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant
from simulation.accountants.standard_dp import StandardDPAccountant
from simulation.calibration import q_from_epsilon, set_threshold_for_hit_rate
from simulation.datasets import generate_synthetic_population, PopulationConfig

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Section 1: Parameters

In [ ]:
# ── Privacy parameters ────────────────────────────────────────────────────────
EPSILON          = 0.01
DELTA            = 1e-6

# ── Non-TCT query model ───────────────────────────────────────────────────────
LAMBDA_NONTCT   = 3.0       # avg non-TCT queries per TCT round (Poisson)
EPS_NONTCT_MIN  = 0.005     # min ε cost per non-TCT query (Uniform draw)
EPS_NONTCT_MAX  = 0.02      # max ε cost per non-TCT query

# ── Population / simulation parameters ───────────────────────────────────────
N_USERS         = 300
N_ROUNDS        = 300
EPS_BUDGET      = 1.0
DELTA_BUDGET    = 1.0
TARGET_HIT_RATE = 0.05
POP_ALPHA       = 0.3
POP_BETA        = 3.0
P_PARTICIPATE   = 0.3      # fraction sampled per query — applies to both TCT and non-TCT rounds
SEED            = 42

q = q_from_epsilon(EPSILON)

config = PopulationConfig(
    n_users=N_USERS, alpha=POP_ALPHA, beta=POP_BETA,
    epsilon_budget=EPS_BUDGET, delta_budget=DELTA_BUDGET,
)

# Calibrate threshold with partial participation so hit rate matches at runtime
calib_rng = np.random.default_rng(SEED)
calib_pop = generate_synthetic_population(config, rng=np.random.default_rng(SEED))
n_pilot = 2000
pilot_counts = np.empty(n_pilot)
for i in range(n_pilot):
    active = calib_pop.active_users
    participants = [u for u in active if calib_rng.random() < P_PARTICIPATE]
    if not participants:
        pilot_counts[i] = calib_rng.laplace(0.0, 1.0 / EPSILON)
        continue
    from simulation.query import compute_round_conversions as _crc
    cnt, _ = _crc(participants, calib_rng)
    pilot_counts[i] = cnt + calib_rng.laplace(0.0, 1.0 / EPSILON)
THRESHOLD = float(np.quantile(pilot_counts, 1.0 - TARGET_HIT_RATE))
actual_hr = float(np.mean(pilot_counts >= THRESHOLD))

mean_nontct_eps = LAMBDA_NONTCT * (EPS_NONTCT_MIN + EPS_NONTCT_MAX) / 2

print(f'ε={EPSILON}, δ={DELTA}, q={q:.4f}, p_participate={P_PARTICIPATE}')
print(f'Threshold T={THRESHOLD:.3f},  empirical hit rate={actual_hr:.3f}')
print(f'Non-TCT model: Poisson(λ={LAMBDA_NONTCT}) queries/round, cost ~ Uniform({EPS_NONTCT_MIN}, {EPS_NONTCT_MAX})')
print(f'Expected non-TCT queries over {N_ROUNDS} rounds: {LAMBDA_NONTCT * N_ROUNDS:.0f}')
print(f'Expected non-TCT ε per round per user (independent): {mean_nontct_eps * P_PARTICIPATE:.4f}')
print(f'Standard IDP expected exhaustion ≈ {int(EPS_BUDGET/(P_PARTICIPATE*EPSILON))} TCT rounds')

## Section 2: Single Run — Retention and Budget Curves

In [ ]:
def run_interleaved(accountant, lambda_nontct, eps_min, eps_max,
                    p_participate, independent_nontct, seed=SEED):
    """
    Simulate TCT rounds with Poisson-distributed non-TCT queries.
    Both TCT and non-TCT queries use partial participation (p_participate).

    TCT rounds:
      - P_PARTICIPATE fraction of active users is sampled each round.
      - Only sampled users' conversions count toward the aggregate; only they are charged.
      - Standard IDP: all sampled users pay ε regardless of contribution.
      - Adaptive TCT: sampled users are registered on first appearance; incremental
        charge fires only if they contributed to a top (contributed=True).

    Non-TCT rounds:
      - Poisson(lambda_nontct) queries per TCT round.
      - Each query: cost ~ Uniform(eps_min, eps_max).
      - independent_nontct=True : each user participates independently with p_participate.
      - independent_nontct=False: all active users pay regardless.
    """
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(config, rng=np.random.default_rng(seed))
    contrib_counts = defaultdict(int)
    active_frac, avg_eps = [], []

    for r in range(N_ROUNDS):
        active = pop.active_users
        if not active:
            break

        # ── TCT round: sample participants ────────────────────────────────────
        participants = [u for u in active if rng.random() < p_participate]
        if participants:
            true_count, contributions = compute_round_conversions(participants, rng)
            is_top, _ = laplace_threshold_query(true_count, THRESHOLD, EPSILON, rng)
            for i, user in enumerate(participants):
                if contributions[i]:
                    contrib_counts[user.user_id] += 1
                e, d = accountant.charge(
                    user=user, is_top=is_top,
                    contributed=contributions[i], round_num=r,
                )
                user.epsilon_spent += e
                user.delta_spent   += d

        # ── Non-TCT queries: Poisson count, uniform cost, per-user participation ──
        n_queries = rng.poisson(lambda_nontct)
        for _ in range(n_queries):
            cost = rng.uniform(eps_min, eps_max)
            for user in pop.active_users:
                participates = rng.random() < p_participate
                if not independent_nontct or participates:
                    user.epsilon_spent += cost

        pop.drop_depleted()
        eps_dist = pop.epsilon_remaining_distribution()
        active_frac.append(pop.active_fraction)
        avg_eps.append(float(np.mean(eps_dist)) if eps_dist else 0.0)

    final_eps = {u.user_id: max(u.epsilon_remaining, 0.0) for u in pop.all_users}
    return active_frac, avg_eps, dict(contrib_counts), final_eps


def half_life(active_frac):
    for r, f in enumerate(active_frac):
        if f <= 0.5:
            return r
    return len(active_frac)


print('Running simulations...')
tct_act, tct_eps, tct_contrib, tct_final = run_interleaved(
    AdaptiveTCTAccountant(EPSILON, DELTA, independent=True),
    LAMBDA_NONTCT, EPS_NONTCT_MIN, EPS_NONTCT_MAX, P_PARTICIPATE, independent_nontct=True)

idp_act, idp_eps, idp_contrib, idp_final = run_interleaved(
    StandardDPAccountant(EPSILON),   # independent=False: all sampled users pay
    LAMBDA_NONTCT, EPS_NONTCT_MIN, EPS_NONTCT_MAX, P_PARTICIPATE, independent_nontct=False)

print('Done.')

In [ ]:
def pad(curve, n):
    return list(curve) + [curve[-1]] * (n - len(curve))

n = max(len(tct_act), len(idp_act)) + 1
rnds = list(range(n))

hl_tct = half_life(tct_act)
hl_idp = half_life(idp_act)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(rnds, pad(tct_act, n), color='steelblue', lw=2,
        label=f'Adaptive TCT (independent)  hl={hl_tct}')
ax.plot(rnds, pad(idp_act, n), color='gray',      lw=2, linestyle='--',
        label=f'Standard IDP                hl={hl_idp}')
ax.axhline(0.5, color='silver', lw=1, linestyle=':')
ax.set_xlabel('TCT Round')
ax.set_ylabel('Fraction of users active')
ax.set_title('User Retention')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(rnds, pad(tct_eps, n), color='steelblue', lw=2, label='Adaptive TCT (independent)')
ax.plot(rnds, pad(idp_eps, n), color='gray',      lw=2, linestyle='--', label='Standard IDP')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('TCT Round')
ax.set_ylabel('Avg ε remaining (active users)')
ax.set_title('Average Budget Remaining')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'N={N_USERS}, ε={EPSILON}, δ={DELTA}  |  '
    f'non-TCT: Poisson(λ={LAMBDA_NONTCT})/round, cost~Uniform({EPS_NONTCT_MIN},{EPS_NONTCT_MAX}), p={P_PARTICIPATE}',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'Half-life: Adaptive TCT (ind)={hl_tct},  Standard IDP={hl_idp}')
print(f'TCT vs IDP advantage: {hl_tct - hl_idp:+d} rounds')

## Section 3: Effect of Participation Probability

How does the independence advantage change as non-TCT participation becomes more or less likely?

- **p=1.0**: every user participates in every non-TCT query — independent and non-independent
  non-TCT costs are identical; advantage comes only from TCT independence.
- **p→0**: non-TCT cost vanishes for independent accountants; TCT independence
  dominates the comparison.

In [ ]:
p_sweep = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

hl_tct_sweep = []
hl_idp_sweep = []

for p in p_sweep:
    a, *_ = run_interleaved(AdaptiveTCTAccountant(EPSILON, DELTA, independent=True),
                            LAMBDA_NONTCT, EPS_NONTCT_MIN, EPS_NONTCT_MAX, p, independent_nontct=True)
    b, *_ = run_interleaved(StandardDPAccountant(EPSILON),   # independent=False
                            LAMBDA_NONTCT, EPS_NONTCT_MIN, EPS_NONTCT_MAX, p, independent_nontct=False)
    hl_tct_sweep.append(half_life(a))
    hl_idp_sweep.append(half_life(b))

gap = [hl_tct_sweep[i] - hl_idp_sweep[i] for i in range(len(p_sweep))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(p_sweep, hl_tct_sweep, 'o-', color='steelblue', lw=2, label='Adaptive TCT (independent)')
ax.plot(p_sweep, hl_idp_sweep, '^--', color='gray',     lw=2, label='Standard IDP')
ax.set_xlabel('p_participate (non-TCT query)')
ax.set_ylabel('Half-life (TCT rounds)')
ax.set_title('Half-Life vs Participation Probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(p_sweep, gap, 'o-', color='steelblue', lw=2)
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('p_participate (non-TCT query)')
ax.set_ylabel('TCT half-life advantage over IDP (rounds)')
ax.set_title('Adaptive TCT Independence Advantage vs Participation Probability')
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'ε={EPSILON}, δ={DELTA}  |  non-TCT: Poisson(λ={LAMBDA_NONTCT}), cost~Uniform({EPS_NONTCT_MIN},{EPS_NONTCT_MAX})',
    fontsize=11
)
plt.tight_layout()
plt.show()

header = '{:>6}  {:>8}  {:>8}  {:>12}'.format('p', 'TCT HL', 'IDP HL', 'TCT−IDP gap')
print(header)
print('-' * 40)
for i, p in enumerate(p_sweep):
    print(f'{p:>6.1f}  {hl_tct_sweep[i]:>8}  {hl_idp_sweep[i]:>8}  {gap[i]:>+12}')

## Section 4: Per-User Cost Stratification by Contribution Count

Under independent accounting, privacy cost should concentrate on users who
frequently contributed to queries.  Users who rarely contributed should retain
most of their budget.

Contribution count here is the number of TCT rounds in which the user personally
generated a conversion (independent of whether the round was a top).

In [ ]:
def stratify_by_contrib(contrib_counts, final_eps):
    buckets = defaultdict(list)
    for uid, eps in final_eps.items():
        count = contrib_counts.get(uid, 0)
        if count == 0:       bucket = '0'
        elif count <= 5:     bucket = '1-5'
        elif count <= 20:    bucket = '6-20'
        else:                bucket = '21+'
        buckets[bucket].append(eps)
    return buckets

tct_buckets = stratify_by_contrib(tct_contrib, tct_final)
idp_buckets = stratify_by_contrib(idp_contrib, idp_final)

bucket_labels = ['0', '1-5', '6-20', '21+']

tct_means = [np.mean(tct_buckets[k]) if tct_buckets[k] else 0.0 for k in bucket_labels]
idp_means = [np.mean(idp_buckets[k]) if idp_buckets[k] else 0.0 for k in bucket_labels]
ns        = [len(tct_buckets[k]) for k in bucket_labels]

x_pos = np.arange(len(bucket_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(x_pos - width/2, tct_means, width, color='steelblue', alpha=0.85, label='Adaptive TCT (ind)')
ax.bar(x_pos + width/2, idp_means, width, color='gray',      alpha=0.85, label='Standard IDP')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{lbl}\n(n={ns[i]})' for i, lbl in enumerate(bucket_labels)])
ax.set_ylabel('Mean ε remaining at end of simulation')
ax.set_title('Budget Remaining by Contribution Bucket')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', lw=0.5)
for bar, val in [(b, b.get_height()) for b in ax.patches]:
    if val > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
                ha='center', va='bottom', fontsize=8)

ax = axes[1]
advantage = [tct_means[i] - idp_means[i] for i in range(len(bucket_labels))]
colors = ['steelblue' if v >= 0 else 'salmon' for v in advantage]
ax.bar(x_pos, advantage, color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(bucket_labels)
ax.set_ylabel('Δ ε remaining (Adaptive TCT − IDP)')
ax.set_title('Adaptive TCT Budget Advantage by Contribution Count')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'N={N_USERS}, ε={EPSILON}, δ={DELTA}  |  '
    f'non-TCT: Poisson(λ={LAMBDA_NONTCT}), p={P_PARTICIPATE}  |  budget={EPS_BUDGET}',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'\n{"Bucket":<8}  {"n":>5}  {"TCT ind":>9}  {"IDP":>9}  {"Advantage":>10}')
print('-' * 48)
for i, lbl in enumerate(bucket_labels):
    print(f'{lbl:<8}  {ns[i]:>5}  {tct_means[i]:>9.4f}  {idp_means[i]:>9.4f}  {advantage[i]:>+10.4f}')